# Weekly Sector Rotation — feature diagnostics

Feature research on top of `data/processed/weekly_sector_rotation/feature_panel.csv`
(built and integrity-checked in `feature_selection.ipynb`), run in this order before
any model training:

1. **Descriptive sanity checks** — distributions, missingness, extreme values, regime
   shifts. *(this notebook, current stage)*
2. Univariate predictive tests — Spearman IC vs. next-week excess return.
3. Quantile tests — weak -> strong feature value vs. next-week excess return.
4. Cross-sectional tests — weekly rank-and-hold-forward test.
5. Stability tests — IC/quantile results across subperiods and regimes.
6. Redundancy tests — Spearman correlation matrix, VIF.
7. Incremental model tests — walk-forward ablation, group by group.
8. Block-bootstrap confidence intervals, especially top-2/top-3 portfolio excess return.

No feature is dropped at this stage on the basis of a weak standalone statistic — the
goal here is a complete report per feature, not a filtered shortlist.

## Label: next-week continuous excess return

$ER_{t+1} = R_{sector,t+1} - R_{SPY,t+1}$, used as the continuous research target
throughout feature diagnostics (the eventual classification target,
$\mathbb{1}(ER_{t+1} > 0)$, is derived from this later, not used here — a sector that
beats SPY by 4% and one that beats it by 0.01% must not look identical during
exploratory analysis).

Built as `excess_return_vs_spy_1w`, shifted **backward by one week within each
symbol's own chronological series** — row `t`'s label is week `t+1`'s realized 1-week
excess return, which is already a leak-free, already-computed column; shifting it is a
relabeling, not a new calculation. The most recent week in the panel has no label (its
"next week" hasn't happened) and is dropped from every diagnostic below, not imputed.

## Feature groups (A-I)

Fixed here once so every later ablation stage (6-7) references the same grouping.
`realized_vol_8w` and `trend_dist_from_ma_8w` exist in the panel as validated features
in their own right but fall outside this specific A-I taxonomy (group D and E are
defined over 4w/12w/26w only) — they're carried in the descriptive report below like
every other feature, just not assigned to a lettered ablation group.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[3]
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=["date", "week_end", "actual_last_trading_date",
                                                       "feature_cutoff_timestamp",
                                                       "next_week_first_trading_date",
                                                       "next_week_last_trading_date"])
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)

AUDIT_COLUMNS = [
    "week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
    "next_week_first_trading_date", "next_week_last_trading_date",
]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

print(f"{panel.shape[0]:,} rows, {len(FEATURE_COLUMNS)} feature columns, "
      f"{panel['date'].min().date()} -> {panel['date'].max().date()}")
panel[["date", "symbol", *FEATURE_COLUMNS[:4]]].tail()

12,551 rows, 36 feature columns, 2004-10-01 -> 2026-08-07


,date,symbol,abs_return_1w,abs_return_2w,abs_return_4w,abs_return_8w
12546,2026-07-10,VPU,-0.006935,-0.017480,0.023790,0.038270
12547,2026-07-17,VPU,-0.004843,-0.011744,0.013264,0.000267
12548,2026-07-24,VPU,0.021719,0.016771,-0.001002,0.042203
12549,2026-07-31,VPU,-0.041312,-0.020490,-0.031993,0.000706
12550,2026-08-07,VPU,-0.012342,-0.053144,-0.037264,-0.014360


In [2]:
FEATURE_GROUPS = {
    "A": ["abs_return_1w", "abs_return_2w", "abs_return_4w", "abs_return_8w", "abs_return_12w", "abs_return_26w"],
    "B": ["excess_return_vs_spy_1w", "excess_return_vs_spy_2w", "excess_return_vs_spy_4w",
          "excess_return_vs_spy_8w", "excess_return_vs_spy_12w", "excess_return_vs_spy_26w"],
    "C": ["cross_sectional_rank_1w", "cross_sectional_rank_4w", "cross_sectional_rank_8w", "cross_sectional_rank_12w"],
    "D": ["realized_vol_4w", "realized_vol_12w", "realized_vol_26w"],
    "E": ["trend_dist_from_ma_4w", "trend_dist_from_ma_12w", "trend_dist_from_ma_26w"],
    "F": ["drawdown_dist_from_high_13w", "drawdown_dist_from_high_26w"],
    "G": ["volume_change_1w", "volume_vs_avg_4w", "volume_vs_avg_12w"],
    "H": ["spy_return_1w", "spy_return_4w", "spy_return_12w", "spy_volatility_4w"],
    "I": ["sector_return_dispersion_1w", "average_sector_return_1w", "sectors_outperforming_spy_1w"],
}
GROUP_NAMES = {
    "A": "Absolute momentum", "B": "Relative momentum vs SPY", "C": "Cross-sectional strength",
    "D": "Volatility", "E": "Trend", "F": "Drawdown", "G": "Volume",
    "H": "Market context", "I": "Rotation / breadth",
}

grouped_columns = {c for cols in FEATURE_GROUPS.values() for c in cols}
ungrouped_columns = [c for c in FEATURE_COLUMNS if c not in grouped_columns]

feature_to_group = {c: g for g, cols in FEATURE_GROUPS.items() for c in cols}

assert grouped_columns <= set(FEATURE_COLUMNS), "a FEATURE_GROUPS entry doesn't exist in the panel"
print(f"{len(grouped_columns)} columns assigned to groups A-I; "
      f"{len(ungrouped_columns)} ungrouped: {ungrouped_columns}")

34 columns assigned to groups A-I; 2 ungrouped: ['trend_dist_from_ma_8w', 'realized_vol_8w']


## Build the label

In [3]:
panel["next_week_excess_return"] = panel.groupby("symbol")["excess_return_vs_spy_1w"].shift(-1)

# Sanity: the label must always describe a period starting strictly after this row's own
# feature cutoff -- the panel's own audit columns already prove this at the row level;
# restate it here since this is the label diagnostics research will actually run on.
labeled = panel.dropna(subset=["next_week_excess_return", "next_week_first_trading_date"])
assert (labeled["feature_cutoff_timestamp"] < labeled["next_week_first_trading_date"]).all(), (
    "a label's target period starts before its own feature cutoff -- lookahead"
)

n_unlabeled = panel["next_week_excess_return"].isna().sum()
print(f"{len(panel) - n_unlabeled:,} labeled rows, {n_unlabeled} unlabeled (most recent week per symbol, "
      f"dropped from diagnostics below rather than imputed)")

research = panel.dropna(subset=["next_week_excess_return"]).reset_index(drop=True)
research[["date", "symbol", "excess_return_vs_spy_1w", "next_week_excess_return"]].tail(11)

12,540 labeled rows, 11 unlabeled (most recent week per symbol, dropped from diagnostics below rather than imputed)


,date,symbol,excess_return_vs_spy_1w,next_week_excess_return
12529,2026-05-22,VPU,0.021800,-0.035069
12530,2026-05-29,VPU,-0.035069,0.023031
12531,2026-06-05,VPU,0.023031,-0.005445
12532,2026-06-12,VPU,-0.005445,-0.004413
12533,2026-06-19,VPU,-0.004413,0.054494
12534,2026-06-26,VPU,0.054494,-0.028673
12535,2026-07-03,VPU,-0.028673,-0.017593
12536,2026-07-10,VPU,-0.017593,0.010398
12537,2026-07-17,VPU,0.010398,0.027741
12538,2026-07-24,VPU,0.027741,-0.050659


## Stage 1: descriptive sanity checks

One row per feature (plus the label itself, for reference): count, missingness, central
tendency/spread, and the tail percentiles requested for the eventual per-feature report.
Computed on `research` (the label-dropped panel) so missingness reflects what a model
would actually see.

In [4]:
REPORT_COLUMNS = FEATURE_COLUMNS + ["next_week_excess_return"]


def describe_feature(col: str) -> pd.Series:
    s = research[col]
    return pd.Series(
        {
            "group": feature_to_group.get(col, "label" if col == "next_week_excess_return" else "-"),
            "n": s.notna().sum(),
            "n_missing": s.isna().sum(),
            "pct_missing": s.isna().mean(),
            "mean": s.mean(),
            "std": s.std(),
            "min": s.min(),
            "p1": s.quantile(0.01),
            "p50": s.median(),
            "p99": s.quantile(0.99),
            "max": s.max(),
            "skew": s.skew(),
        }
    )


descriptive = pd.DataFrame({col: describe_feature(col) for col in REPORT_COLUMNS}).T
descriptive.index.name = "feature"
descriptive[["n", "n_missing"]] = descriptive[["n", "n_missing"]].astype(int)

descriptive.to_csv(OUTPUTS_ROOT / "descriptive_stats.csv")
descriptive

,group,n,n_missing,pct_missing,mean,std,min,p1,p50,p99,max,skew
feature,,,,,,,,,,,,
abs_return_1w,A,12529,11,0.000877,0.002231,0.02982,-0.249736,-0.081994,0.003309,0.079114,0.284032,-0.225021
abs_return_2w,A,12518,22,0.001754,0.004414,0.040425,-0.390798,-0.115729,0.006481,0.103865,0.28624,-0.827627
abs_return_4w,A,12496,44,0.003509,0.008867,0.05679,-0.527355,-0.155046,0.01238,0.147113,0.513328,-0.624902
abs_return_8w,A,12452,88,0.007018,0.017416,0.076931,-0.576019,-0.227892,0.022072,0.204063,0.631441,-0.656469
abs_return_12w,A,12408,132,0.010526,0.025801,0.09205,-0.578862,-0.275604,0.030811,0.251882,0.810827,-0.540621
abs_return_26w,A,12254,286,0.022807,0.055963,0.136361,-0.663957,-0.391913,0.063625,0.408451,1.110521,-0.24551
excess_return_vs_spy_1w,B,12529,11,0.000877,-0.000098,0.017814,-0.150957,-0.047591,-0.000106,0.047794,0.174094,0.14416
excess_return_vs_spy_2w,B,12518,22,0.001754,-0.000202,0.024544,-0.223851,-0.063715,-0.000374,0.06737,0.21489,0.163009
excess_return_vs_spy_4w,B,12496,44,0.003509,-0.00044,0.034606,-0.228991,-0.091823,-0.00074,0.094003,0.338815,0.353568


### Missingness pattern

Confirm every NaN is confined to each symbol's own warm-up period (its first 26 weeks,
the longest window used anywhere) rather than showing up sporadically later — a NaN
appearing after warm-up would indicate a data gap or a bug, not an expected condition.

In [5]:
WARMUP_WEEKS = 26

research_sorted = research.sort_values(["symbol", "date"]).copy()
research_sorted["week_position"] = research_sorted.groupby("symbol").cumcount()

late_nan_counts = {}
for col in FEATURE_COLUMNS:
    late_nan = research_sorted.loc[
        research_sorted[col].isna() & (research_sorted["week_position"] >= WARMUP_WEEKS), col
    ]
    if len(late_nan):
        late_nan_counts[col] = len(late_nan)

assert not late_nan_counts, f"unexpected post-warmup NaNs: {late_nan_counts}"
print(f"missingness confirmed confined to each symbol's first {WARMUP_WEEKS} weeks -- "
      f"no unexpected NaNs afterward, across all {len(FEATURE_COLUMNS)} feature columns")

missingness confirmed confined to each symbol's first 26 weeks -- no unexpected NaNs afterward, across all 36 feature columns


### Extreme values

Every feature value expressed as a z-score against its own full-period mean/std; values
beyond |z| > 5 are flagged for manual review (not removed) — the goal here is catching
a bad tick or a construction bug, not winsorizing.

In [6]:
Z_THRESHOLD = 5.0

extreme_rows = []
for col in FEATURE_COLUMNS:
    s = research[col]
    mean, std = s.mean(), s.std()
    if std == 0 or np.isnan(std):
        continue
    z = (s - mean) / std
    flagged = z[z.abs() > Z_THRESHOLD]
    for idx, z_value in flagged.items():
        extreme_rows.append(
            {
                "feature": col,
                "group": feature_to_group.get(col, "-"),
                "date": research.loc[idx, "date"],
                "symbol": research.loc[idx, "symbol"],
                "value": research.loc[idx, col],
                "z_score": z_value,
            }
        )

extreme_observations = pd.DataFrame(extreme_rows).sort_values("z_score", key=abs, ascending=False)
extreme_observations.to_csv(OUTPUTS_ROOT / "extreme_observations.csv", index=False)

print(f"{len(extreme_observations)} observations beyond |z| > {Z_THRESHOLD} "
      f"across {extreme_observations['feature'].nunique() if len(extreme_observations) else 0} features")
extreme_observations.head(20)

1509 observations beyond |z| > 5.0 across 31 features


,feature,group,date,symbol,value,z_score
1152,volume_vs_avg_4w,G,2006-07-07,VOX,27.269205,34.120525
1201,volume_vs_avg_12w,G,2006-01-27,VNQ,23.086398,28.741806
1092,volume_change_1w,G,2006-07-07,VOX,26.430079,27.327789
1091,volume_change_1w,G,2006-01-06,VOX,25.957983,26.836433
1178,volume_vs_avg_12w,G,2006-11-03,VCR,21.120815,26.287330
1222,volume_vs_avg_12w,G,2012-08-03,VPU,20.132100,25.052695
1107,volume_change_1w,G,2012-08-03,VPU,24.216508,25.023908
1162,volume_vs_avg_4w,G,2012-08-03,VPU,19.570741,24.457325
1088,volume_change_1w,G,2006-01-27,VNQ,23.647852,24.432052
1065,volume_change_1w,G,2006-11-03,VCR,23.542453,24.322352


### Regime shifts

A lightweight check, not the formal subperiod IC/quantile analysis planned for Stage 5:
split the panel's date range into 4 equal-length calendar blocks and compare each
feature's mean across blocks against its own full-period std. A high ratio flags a
feature whose *level* drifts across eras (rate regimes, sector composition drift, etc.)
— worth knowing before interpreting a full-period correlation as if it were stationary.

In [7]:
N_BLOCKS = 4

unique_dates = np.array(sorted(research["date"].unique()))
date_blocks = np.array_split(unique_dates, N_BLOCKS)
date_to_block = {d: i for i, block in enumerate(date_blocks) for d in block}
research["regime_block"] = research["date"].map(date_to_block)

block_ranges = {
    i: (pd.Timestamp(block.min()).date(), pd.Timestamp(block.max()).date()) for i, block in enumerate(date_blocks)
}
print("regime blocks:", block_ranges)

block_means = research.groupby("regime_block")[FEATURE_COLUMNS].mean()
full_period_std = research[FEATURE_COLUMNS].std()

regime_instability = (block_means.std() / full_period_std).rename("instability_ratio").to_frame()
regime_instability["group"] = [feature_to_group.get(c, "-") for c in regime_instability.index]
regime_instability = regime_instability.sort_values("instability_ratio", ascending=False)
regime_instability.index.name = "feature"

regime_instability.to_csv(OUTPUTS_ROOT / "regime_instability.csv")
block_means.to_csv(OUTPUTS_ROOT / "regime_block_means.csv")

regime_instability.head(15)

regime blocks: {0: (datetime.date(2004, 10, 1), datetime.date(2010, 3, 12)), 1: (datetime.date(2010, 3, 19), datetime.date(2015, 8, 28)), 2: (datetime.date(2015, 9, 4), datetime.date(2021, 2, 12)), 3: (datetime.date(2021, 2, 19), datetime.date(2026, 7, 31))}


,instability_ratio,group
feature,,
sector_return_dispersion_1w,0.342599,I
realized_vol_26w,0.251673,D
drawdown_dist_from_high_26w,0.231163,F
realized_vol_12w,0.217489,D
realized_vol_8w,0.199108,-
drawdown_dist_from_high_13w,0.171620,F
realized_vol_4w,0.166960,D
spy_return_12w,0.153243,H
abs_return_26w,0.144460,A


## Stage 2: univariate predictive tests (Spearman IC)

**IC** here means the standard cross-sectional factor-research definition, not a single
pooled correlation: for each week `t` independently, Spearman-rank-correlate the
feature's values **across that week's 11 sectors** against those same sectors'
`next_week_excess_return`, giving one IC observation per week. Pooling all
`weeks x symbols` rows into one correlation instead would conflate cross-sectional
signal with time-series level drift (e.g. a feature that trends upward across 2020-2026
for every sector at once would show a spurious pooled correlation with a label that also
drifted, even with zero genuine cross-sectional selection power) — the per-week
cross-sectional design is what "IC" means precisely to rule that out. This is also why
Stage 4 (weekly rank-and-hold) is a separate, complementary test rather than a
duplicate of this one: same weekly cross-section, different statistic.

Every week in the 11-symbol panel has either all 11 sectors defined for a given feature
or none (warm-up NaNs land on the same weeks for every symbol), so no week contributes a
partial, ragged cross-section.

**Sign convention**: for return/trend/volume-style features, a positive IC means "higher
feature value -> higher next-week excess return." For `cross_sectional_rank_*w`
(1 = strongest), the natural relationship point the *other* way — if momentum persists,
expect a **negative** IC there (lower rank number = stronger prior performer = higher
next-week excess return), not a positive one. This isn't a bug, just the rank
direction chosen back in `feature_selection.ipynb`.

In [8]:
def weekly_ics(group: pd.DataFrame) -> pd.Series:
    """One week's cross-sectional Spearman IC for every feature, computed across its 11 sectors."""
    label = group["next_week_excess_return"]
    result = {}
    for col in FEATURE_COLUMNS:
        feature = group[col]
        if feature.isna().any() or label.isna().any() or feature.std() == 0 or label.std() == 0:
            result[col] = np.nan
            continue
        result[col] = feature.corr(label, method="spearman")
    return pd.Series(result)


ic_by_week = research.groupby("date")[FEATURE_COLUMNS + ["next_week_excess_return"]].apply(weekly_ics)
ic_by_week.to_csv(OUTPUTS_ROOT / "ic_by_week.csv")

n_weeks_total = len(ic_by_week)
n_weeks_used = ic_by_week.notna().sum()
print(f"{n_weeks_total} candidate weeks; per-feature weeks with a full 11-sector cross-section: "
      f"{n_weeks_used.min()}-{n_weeks_used.max()}")
ic_by_week.tail()

/Users/gavinhussey/Downloads/rcm/scrapyard_project/.venv/lib/python3.14/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


1140 candidate weeks; per-feature weeks with a full 11-sector cross-section: 0-1139


,abs_return_1w,abs_return_2w,abs_return_4w,abs_return_8w,abs_return_12w,abs_return_26w,excess_return_vs_spy_1w,excess_return_vs_spy_2w,excess_return_vs_spy_4w,excess_return_vs_spy_8w,...,volume_change_1w,volume_vs_avg_4w,volume_vs_avg_12w,spy_return_1w,spy_return_4w,spy_return_12w,spy_volatility_4w,sector_return_dispersion_1w,sectors_outperforming_spy_1w,average_sector_return_1w
date,,,,,,,,,,,,,,,,,,,,,
2026-07-03,0.000000,-0.281818,-0.572727,-0.436364,-0.100000,0.027273,0.000000,-0.281818,-0.572727,-0.436364,...,-0.027273,-0.190909,-0.209091,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-07-10,0.045455,-0.154545,-0.254545,0.009091,0.081818,0.081818,0.045455,-0.154545,-0.254545,0.009091,...,-0.136364,-0.009091,-0.072727,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-07-17,0.281818,-0.118182,0.127273,-0.036364,-0.045455,0.618182,0.281818,-0.118182,0.127273,-0.036364,...,0.481818,0.100000,-0.136364,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-07-24,-0.863636,-0.654545,-0.327273,-0.318182,-0.236364,-0.572727,-0.863636,-0.654545,-0.327273,-0.318182,...,0.272727,0.200000,0.281818,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-07-31,0.181818,-0.218182,-0.618182,-0.563636,-0.372727,-0.154545,0.181818,-0.218182,-0.618182,-0.563636,...,0.054545,0.281818,0.081818,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### IC summary: full-period, by subperiod, and hit rate

Reuses the same 4 regime blocks from Stage 1 for "IC by subperiod," so the two stages
stay directly comparable. "% periods with correct sign" is the share of weeks whose IC
shares the sign of that feature's own full-period mean IC (not an externally assumed
sign) — a feature can have a small full-period IC but still be highly *consistent*
(high % correct sign), or a larger full-period IC driven by a few volatile weeks (low %
correct sign); the two numbers answer different questions and both matter before Stage
5's deeper stability analysis.

In [9]:
full_period_ic = ic_by_week.mean()
full_period_ic_std = ic_by_week.std()

ic_block = ic_by_week.copy()
ic_block["regime_block"] = ic_block.index.map(date_to_block)
ic_by_subperiod = ic_block.groupby("regime_block")[FEATURE_COLUMNS].mean().T
ic_by_subperiod.columns = [f"ic_block{i}" for i in ic_by_subperiod.columns]

pct_correct_sign = {}
for col in FEATURE_COLUMNS:
    weekly = ic_by_week[col].dropna()
    ref_sign = np.sign(full_period_ic[col])
    if weekly.empty or ref_sign == 0 or np.isnan(ref_sign):
        pct_correct_sign[col] = np.nan
        continue
    pct_correct_sign[col] = (np.sign(weekly) == ref_sign).mean()

ic_summary = pd.DataFrame(
    {
        "group": [feature_to_group.get(c, "-") for c in FEATURE_COLUMNS],
        "n_weeks": n_weeks_used,
        "full_period_ic": full_period_ic,
        "full_period_ic_std": full_period_ic_std,
        "pct_periods_correct_sign": pd.Series(pct_correct_sign),
    }
).join(ic_by_subperiod)

ic_summary = ic_summary.sort_values("full_period_ic", key=abs, ascending=False)
ic_summary.index.name = "feature"

ic_summary.to_csv(OUTPUTS_ROOT / "ic_summary.csv")
ic_summary

,group,n_weeks,full_period_ic,full_period_ic_std,pct_periods_correct_sign,ic_block0,ic_block1,ic_block2,ic_block3
feature,,,,,,,,,
abs_return_1w,A,1139,-0.015085,0.405498,0.506585,-0.007939,-0.042967,0.008485,-0.017895
excess_return_vs_spy_1w,B,1139,-0.015085,0.405498,0.506585,-0.007939,-0.042967,0.008485,-0.017895
cross_sectional_rank_1w,C,1139,0.015085,0.405498,0.506585,0.007939,0.042967,-0.008485,0.017895
abs_return_4w,A,1136,-0.012292,0.412956,0.504401,0.015464,-0.010080,-0.028262,-0.025901
cross_sectional_rank_4w,C,1136,0.012292,0.412956,0.504401,-0.015464,0.010080,0.028262,0.025901
excess_return_vs_spy_4w,B,1136,-0.012292,0.412956,0.504401,0.015464,-0.010080,-0.028262,-0.025901
excess_return_vs_spy_12w,B,1128,-0.011259,0.412467,0.507979,0.015851,0.001722,-0.033620,-0.027847
abs_return_12w,A,1128,-0.011259,0.412467,0.507979,0.015851,0.001722,-0.033620,-0.027847
cross_sectional_rank_12w,C,1128,0.011259,0.412467,0.507979,-0.015851,-0.001722,0.033620,0.027847


### Finding: groups H and I are structurally untestable by cross-sectional IC

`spy_return_1w`, `spy_return_4w`, `spy_return_12w`, `spy_volatility_4w`,
`sector_return_dispersion_1w`, `average_sector_return_1w`, and
`sectors_outperforming_spy_1w` all come back `n_weeks = 0`, `IC = NaN` above — not a
bug. Every one of these is a **date-level** value, broadcast identically to all 11
sector rows for that week by construction (`feature_selection.ipynb`'s `market_context`/
`breadth` DataFrames). A within-week cross-sectional rank correlation needs the feature
to actually vary *across* the 11 sectors that week; these features have exactly one
value across all 11, every week, so their cross-sectional standard deviation is 0 and
the correlation is undefined (0/0), not weak.

This isn't specific to the IC test — Stage 3 (quantile buckets) and Stage 4 (weekly
rank-and-hold) both also rank sectors *within* a week, so they'd hit the same wall for
these 7 features for the same structural reason. A market-level or breadth feature
can't tell you *which* sector to prefer this week (it says nothing that differs sector
to sector) — its only possible predictive content is a **time-series** relationship
(e.g. "does this week's `spy_volatility_4w` level predict next week's *average* sector
excess return, or the *dispersion* of next week's sector returns, pooled across weeks
rather than ranked within them"), which is a different test than anything in the
stage 2-4 cross-sectional design as originally scoped.

## Stage 3: quantile tests

Each week, the 11 sectors are split into **terciles** by that week's feature value
(bottom/middle/top, ranked low -> high, sized ~4/4/3 via `qcut` on rank position — with
only 11 assets per week, quintiles would leave ~2 per bucket, too thin to read).
Group H/I are skipped, per the Stage 2 finding: with zero cross-sectional variance
within a week, every sector would land in the same bucket every week, which isn't a
quantile split at all.

Bucket membership is pooled across all weeks before averaging (`next_week_excess_return`
across every (week, sector) observation that fell in a given bucket, not an average of
per-week bucket means) — the standard construction for a quantile-spread table.

**Reading the sign**: buckets are always ranked by *raw* feature value, low -> high, with
no per-feature direction correction. A feature where higher values genuinely predict
outperformance should show top > bottom; `cross_sectional_rank_*w` (1 = strongest) is
expected to show the mirror image — bottom > top — exactly matching its negative IC from
Stage 2, not a contradiction.

In [10]:
TESTABLE_FEATURES = [c for c in FEATURE_COLUMNS if c not in FEATURE_GROUPS["H"] and c not in FEATURE_GROUPS["I"]]
BUCKET_LABELS = ["bottom", "middle", "top"]


def assign_weekly_terciles(s: pd.Series) -> pd.Series:
    """Bucket one week's 11 sector values into terciles by rank; NaN in -> NaN out (warmup)."""
    if s.isna().any():
        return pd.Series(np.nan, index=s.index)
    ranks = s.rank(method="first")
    return pd.qcut(ranks, 3, labels=BUCKET_LABELS)


quantile_rows = []
for col in TESTABLE_FEATURES:
    bucket = research.groupby("date")[col].apply(assign_weekly_terciles).reset_index(level=0, drop=True)
    valid = bucket.notna()
    sub = research.loc[valid, ["date", "symbol", "next_week_excess_return"]].copy()
    sub["feature"] = col
    sub["bucket"] = bucket[valid].values
    quantile_rows.append(sub)

quantile_long = pd.concat(quantile_rows, ignore_index=True)
quantile_long.to_csv(OUTPUTS_ROOT / "quantile_bucket_assignments.csv", index=False)

print(f"{len(TESTABLE_FEATURES)} testable features (H/I excluded), "
      f"{len(quantile_long):,} pooled (week, sector, feature) bucket observations")
quantile_long.groupby(["feature", "bucket"], observed=True).size().unstack("bucket")[BUCKET_LABELS].head()

29 testable features (H/I excluded), 360,569 pooled (week, sector, feature) bucket observations


bucket,bottom,middle,top
feature,,,
abs_return_12w,4512,3384,4512
abs_return_1w,4556,3417,4556
abs_return_26w,4456,3342,4456
abs_return_2w,4552,3414,4552
abs_return_4w,4544,3408,4544


### Quantile summary: bottom/top mean, spread, monotonicity

`monotonic` is `True` when the middle bucket's mean sits between bottom and top (in
either direction) — a "sensible" weak -> strong progression, not just a strong top vs.
bottom gap with a middle bucket that doesn't fit the story.

In [11]:
bucket_stats = quantile_long.groupby(["feature", "bucket"], observed=True)["next_week_excess_return"].agg(
    ["mean", "std", "count"]
)
bucket_means = bucket_stats["mean"].unstack("bucket")[BUCKET_LABELS]
bucket_counts = bucket_stats["count"].unstack("bucket")[BUCKET_LABELS]

quantile_summary = pd.DataFrame(
    {
        "group": [feature_to_group.get(f, "-") for f in bucket_means.index],
        "bottom_mean": bucket_means["bottom"],
        "middle_mean": bucket_means["middle"],
        "top_mean": bucket_means["top"],
        "top_minus_bottom_spread": bucket_means["top"] - bucket_means["bottom"],
        "n_bottom": bucket_counts["bottom"].astype(int),
        "n_top": bucket_counts["top"].astype(int),
    },
    index=bucket_means.index,
)
quantile_summary.index.name = "feature"

ordered = bucket_means[["bottom", "middle", "top"]]
quantile_summary["monotonic"] = (
    ordered["bottom"].le(ordered["middle"]) & ordered["middle"].le(ordered["top"])
) | (ordered["bottom"].ge(ordered["middle"]) & ordered["middle"].ge(ordered["top"]))

quantile_summary = quantile_summary.sort_values("top_minus_bottom_spread", key=abs, ascending=False)
quantile_summary.to_csv(OUTPUTS_ROOT / "quantile_summary.csv")
quantile_summary

,group,bottom_mean,middle_mean,top_mean,top_minus_bottom_spread,n_bottom,n_top,monotonic
feature,,,,,,,,
excess_return_vs_spy_4w,B,-0.000454,-0.000143,0.000239,0.000693,4544,4544,True
cross_sectional_rank_4w,C,0.000239,-0.000143,-0.000454,-0.000693,4544,4544,True
abs_return_4w,A,-0.000454,-0.000143,0.000239,0.000693,4544,4544,True
cross_sectional_rank_8w,C,0.000217,-0.000132,-0.000459,-0.000676,4528,4528,True
abs_return_8w,A,-0.000459,-0.000132,0.000217,0.000676,4528,4528,True
excess_return_vs_spy_8w,B,-0.000459,-0.000132,0.000217,0.000676,4528,4528,True
trend_dist_from_ma_8w,-,-0.000419,-0.000135,0.000194,0.000613,4532,4532,True
cross_sectional_rank_1w,C,-0.000004,0.000341,-0.000579,-0.000575,4556,4556,False
excess_return_vs_spy_1w,B,-0.000579,0.000341,-0.000004,0.000575,4556,4556,False


## Stage 4: cross-sectional rank-and-hold test

Different from Stage 3's pooled tercile-bucket averages: this builds an actual **weekly
portfolio spread time series** — each week, rank the 11 sectors by that week's feature
value, take the mean `next_week_excess_return` of the top-K and bottom-K sectors
(K = 2 and 3, matching the top-2/top-3 portfolios named for Stage 8's later
block-bootstrap, which will reuse this exact weekly spread series), and record
`spread = top_K_return - bottom_K_return` for that week. Stage 3 asks "what's the
average outcome across all sector-weeks in a bucket" (a single pooled number); this asks
"if you mechanically held this rank-based portfolio every single week, what does *that*
week-by-week series look like" — the second question is what a hit rate, a Sharpe-like
ratio, and eventually a bootstrap confidence interval need, since none of those are
well-defined on pooled cross-sectional observations that mix within-week and
across-week variation.

H/I are skipped for the same structural reason as Stages 2-3. Top/bottom-K selection
uses the same raw-value, no-direction-correction convention as Stage 3, so
`cross_sectional_rank_*w` is expected to show a negative mean spread (mirroring its
negative IC), not a positive one.

In [12]:
RANK_K_VALUES = [2, 3]


def weekly_top_bottom_returns(group: pd.DataFrame, col: str, k: int) -> pd.Series:
    """One week's equal-weighted top-K and bottom-K next-week excess return, ranked by col."""
    if group[col].isna().any():
        return pd.Series({"top_return": np.nan, "bottom_return": np.nan})
    ranked = group[col].rank(ascending=False, method="first")  # 1 = highest raw value that week
    n = len(group)
    top_return = group.loc[ranked <= k, "next_week_excess_return"].mean()
    bottom_return = group.loc[ranked > n - k, "next_week_excess_return"].mean()
    return pd.Series({"top_return": top_return, "bottom_return": bottom_return})


spread_frames = []
for k in RANK_K_VALUES:
    for col in TESTABLE_FEATURES:
        weekly = research.groupby("date").apply(weekly_top_bottom_returns, col=col, k=k)
        weekly["feature"] = col
        weekly["k"] = k
        spread_frames.append(weekly.reset_index())

rank_hold = pd.concat(spread_frames, ignore_index=True)
rank_hold["spread"] = rank_hold["top_return"] - rank_hold["bottom_return"]
rank_hold = rank_hold.dropna(subset=["spread"]).reset_index(drop=True)
rank_hold.to_csv(OUTPUTS_ROOT / "rank_hold_weekly_spread.csv", index=False)

print(f"{len(TESTABLE_FEATURES)} features x {len(RANK_K_VALUES)} K values, "
      f"{len(rank_hold):,} weekly (feature, k) spread observations")
rank_hold[rank_hold["feature"] == "excess_return_vs_spy_4w"].tail()

29 features x 2 K values, 65,558 weekly (feature, k) spread observations


,date,top_return,bottom_return,feature,k,spread
42974,2026-07-03,-0.019805,0.017069,excess_return_vs_spy_4w,3,-0.036874
42975,2026-07-10,0.012663,0.029430,excess_return_vs_spy_4w,3,-0.016767
42976,2026-07-17,0.018868,0.016053,excess_return_vs_spy_4w,3,0.002816
42977,2026-07-24,-0.015927,0.006736,excess_return_vs_spy_4w,3,-0.022663
42978,2026-07-31,-0.042446,-0.010191,excess_return_vs_spy_4w,3,-0.032254


### Rank-and-hold summary

`sharpe_like` is `mean_spread / std_spread * sqrt(52)` — an annualized-scale diagnostic
for *comparing features against each other*, not a claim about a tradeable strategy
(no costs, no sizing, no borrow, equal-weighted K-legs only). Real risk-adjusted
performance estimation is Stage 8's job.

In [13]:
rank_hold_summary = rank_hold.groupby(["feature", "k"]).agg(
    mean_spread=("spread", "mean"),
    std_spread=("spread", "std"),
    n_weeks=("spread", "count"),
    hit_rate=("spread", lambda s: (s > 0).mean()),
).reset_index()

rank_hold_summary["sharpe_like"] = (
    rank_hold_summary["mean_spread"] / rank_hold_summary["std_spread"] * np.sqrt(52)
)
rank_hold_summary["group"] = rank_hold_summary["feature"].map(lambda f: feature_to_group.get(f, "-"))
rank_hold_summary = rank_hold_summary.sort_values("mean_spread", key=abs, ascending=False)

rank_hold_summary.to_csv(OUTPUTS_ROOT / "rank_hold_summary.csv", index=False)
rank_hold_summary.head(20)

,feature,k,mean_spread,std_spread,n_weeks,hit_rate,sharpe_like,group
26,excess_return_vs_spy_1w,2,-0.001033,0.026337,1139,0.489025,-0.282919,B
2,abs_return_1w,2,-0.001033,0.026337,1139,0.489025,-0.282919,A
14,cross_sectional_rank_1w,2,0.001033,0.026337,1139,0.510975,0.282919,C
8,abs_return_4w,2,-0.000910,0.025273,1136,0.480634,-0.259522,A
16,cross_sectional_rank_4w,2,0.000910,0.025273,1136,0.519366,0.259522,C
32,excess_return_vs_spy_4w,2,-0.000910,0.025273,1136,0.480634,-0.259522,B
24,excess_return_vs_spy_12w,2,-0.000821,0.025775,1128,0.500000,-0.229754,B
12,cross_sectional_rank_12w,2,0.000821,0.025775,1128,0.500000,0.229754,C
0,abs_return_12w,2,-0.000821,0.025775,1128,0.500000,-0.229754,A
13,cross_sectional_rank_12w,3,0.000804,0.021054,1128,0.487589,0.275382,C


### Finding: extreme top/bottom-K selection reverses sign vs. Stage 3's broader terciles

Several of the features with the strongest, most consistent Stage 2-3 results —
`abs_return_1w/4w/12w`, `excess_return_vs_spy_1w/4w/12w`, `cross_sectional_rank_1w/4w/12w`
— flip sign here. Stage 3's tercile spread (top ~4 sectors vs. bottom ~4) was positive
for the return-based versions of these features (momentum continuation); Stage 4's
tight top-2 and top-3 rank-and-hold spread for the *same* features is **negative** —
manually verified above on 2015-06-05, not a sign-convention bug.

This means: broadening the "long" group to the top third of sectors by recent momentum
shows mild continuation, but narrowing it to only the single strongest or two/three
strongest sectors shows mild **reversal** instead — the most extreme short-term movers
tend to underperform the next week, not continue outperforming. This is directly
decision-relevant, not academic: a top-2/top-3 concentrated portfolio (the sizing
`Stage 8` is scoped around) built on these particular momentum features would have
historically leaned into the wrong side of that reversal effect, at these K values.
`cross_sectional_rank_*` inherits the same reversal in mirrored form.

Not every feature reverses this way — `volume_change_1w` and `realized_vol_12w/26w`
keep the *same* sign and improve in relative strength at K=2/3 vs. their weaker Stage
2-3 showing, worth noting for the eventual model rather than assuming momentum features
are automatically the best top-K candidates. This is a finding to carry into Stage 5
(does the reversal hold across regimes, or is it period-specific) and Stage 7 (does a
model combining momentum with other groups override this effect) — not a reason to
discard the momentum features now, since Stage 3 says they still carry real, monotonic
signal at a less concentrated portfolio size.

## Stage 5: stability tests

Repeats Stage 2's IC and Stage 3's quantile-spread tests, sliced across three
different period/regime definitions, reusing `ic_by_week` and the pooled
`quantile_long` bucket assignments already computed — not recomputed from the raw
panel — so this stage is purely "does the same signal hold up when you look at
different slices of time," not a new measurement methodology.

**Chronological blocks**: the same 4 calendar blocks from Stage 1 (already have
`date_to_block`).

**Market volatility regime**: `high_vol` / `low_vol`, a median split of
`spy_volatility_4w` — this is the first real use for a group H feature in this
notebook; it was unusable as a cross-sectional factor (Stages 2-4) because it's
constant across all 11 sectors each week, but that's exactly what makes it a valid
week-level regime label.

**Market trend regime**: `bull` / `bear`, by the sign of `spy_return_12w`.

For each feature x regime scheme, `*_sign_consistent` is `True` only when every group
in that scheme shares the same sign (matching the full-period IC's sign, for the IC
table) — a feature whose sign flips between regimes is a materially different claim
than one that's merely weaker in one regime than another.

Scope note: this repeats the IC/quantile methodology as specified — it does not repeat
Stage 4's rank-and-hold test by regime. Given Stage 4 found top-2/3 reversal in several
momentum features, checking whether *that specific* result is regime-dependent would be
a natural follow-up, but it's a different test than what's built here; flagging it
rather than silently expanding this stage's scope.

In [14]:
date_context = (
    research.drop_duplicates(subset="date")
    .set_index("date")[["spy_volatility_4w", "spy_return_12w"]]
    .sort_index()
)

vol_median = date_context["spy_volatility_4w"].median()
date_context["vol_regime"] = np.where(date_context["spy_volatility_4w"] >= vol_median, "high_vol", "low_vol")
date_context["trend_regime"] = np.where(date_context["spy_return_12w"] >= 0, "bull", "bear")
date_context["chron_block"] = date_context.index.map(date_to_block)

REGIME_SCHEMES = {
    "chron_block": sorted(date_context["chron_block"].dropna().unique()),
    "vol_regime": ["low_vol", "high_vol"],
    "trend_regime": ["bear", "bull"],
}

print(date_context["vol_regime"].value_counts().to_dict())
print(date_context["trend_regime"].value_counts().to_dict())
print(date_context["chron_block"].value_counts().sort_index().to_dict())
date_context.tail()

{'low_vol': 572, 'high_vol': 568}
{'bull': 824, 'bear': 316}
{0: 285, 1: 285, 2: 285, 3: 285}


,spy_volatility_4w,spy_return_12w,vol_regime,trend_regime,chron_block
date,,,,,
2026-07-03,0.015745,0.103778,low_vol,bull,3
2026-07-10,0.015985,0.066277,low_vol,bull,3
2026-07-17,0.018243,0.045493,high_vol,bull,3
2026-07-24,0.015191,0.029720,low_vol,bull,3
2026-07-31,0.012526,0.017635,low_vol,bull,3


### IC stability across chronological blocks and market regimes

In [15]:
ic_stability_rows = []
for col in TESTABLE_FEATURES:
    row = {"feature": col, "group": feature_to_group.get(col, "-"), "full_period_ic": full_period_ic[col]}
    for scheme, groups in REGIME_SCHEMES.items():
        merged = ic_by_week[[col]].join(date_context[scheme])
        group_means = merged.groupby(scheme, observed=True)[col].mean()
        for g in groups:
            row[f"{scheme}={g}"] = group_means.get(g, np.nan)
        signs = np.sign(group_means.dropna())
        ref_sign = np.sign(full_period_ic[col])
        row[f"{scheme}_sign_consistent"] = bool((signs == ref_sign).all()) if len(signs) else np.nan
    ic_stability_rows.append(row)

ic_stability = pd.DataFrame(ic_stability_rows).set_index("feature")
ic_stability.to_csv(OUTPUTS_ROOT / "ic_stability_by_regime.csv")
ic_stability

,group,full_period_ic,chron_block=0,chron_block=1,chron_block=2,chron_block=3,chron_block_sign_consistent,vol_regime=low_vol,vol_regime=high_vol,vol_regime_sign_consistent,trend_regime=bear,trend_regime=bull,trend_regime_sign_consistent
feature,,,,,,,,,,,,,
abs_return_1w,A,-0.015085,-0.007939,-0.042967,0.008485,-0.017895,False,-0.013963,-0.016213,True,-0.047215,-0.002802,True
abs_return_2w,A,-0.006399,0.010215,0.001722,-0.003573,-0.033844,False,-0.013285,0.000512,False,-0.008686,-0.005527,True
abs_return_4w,A,-0.012292,0.015464,-0.010080,-0.028262,-0.025901,False,-0.007378,-0.017206,True,-0.037413,-0.002780,True
abs_return_8w,A,-0.009741,0.012406,-0.008389,-0.030686,-0.011675,False,-0.018182,-0.001241,True,-0.015643,-0.007535,True
abs_return_12w,A,-0.011259,0.015851,0.001722,-0.033620,-0.027847,False,-0.003443,-0.019102,True,-0.025598,-0.005969,True
abs_return_26w,A,0.005819,0.029800,-0.001244,0.019139,-0.022233,False,0.004471,0.007146,True,-0.005629,0.010076,False
excess_return_vs_spy_1w,B,-0.015085,-0.007939,-0.042967,0.008485,-0.017895,False,-0.013963,-0.016213,True,-0.047215,-0.002802,True
excess_return_vs_spy_2w,B,-0.006399,0.010215,0.001722,-0.003573,-0.033844,False,-0.013285,0.000512,False,-0.008686,-0.005527,True
excess_return_vs_spy_4w,B,-0.012292,0.015464,-0.010080,-0.028262,-0.025901,False,-0.007378,-0.017206,True,-0.037413,-0.002780,True


### Quantile (top-minus-bottom) stability across the same blocks and regimes

In [16]:
q_merged = quantile_long.merge(
    date_context.reset_index()[["date", "chron_block", "vol_regime", "trend_regime"]], on="date", how="left"
)

quantile_stability_rows = []
for col in TESTABLE_FEATURES:
    row = {"feature": col, "group": feature_to_group.get(col, "-"),
           "full_period_spread": quantile_summary.loc[col, "top_minus_bottom_spread"]}
    sub = q_merged[q_merged["feature"] == col]
    for scheme, groups in REGIME_SCHEMES.items():
        spreads = {}
        for g in groups:
            gsub = sub[sub[scheme] == g]
            top = gsub.loc[gsub["bucket"] == "top", "next_week_excess_return"].mean()
            bottom = gsub.loc[gsub["bucket"] == "bottom", "next_week_excess_return"].mean()
            spreads[g] = top - bottom
            row[f"{scheme}={g}_spread"] = spreads[g]
        valid_signs = {np.sign(v) for v in spreads.values() if pd.notna(v) and v != 0}
        row[f"{scheme}_sign_consistent"] = bool(len(valid_signs) <= 1) if valid_signs else np.nan
    quantile_stability_rows.append(row)

quantile_stability = pd.DataFrame(quantile_stability_rows).set_index("feature")
quantile_stability.to_csv(OUTPUTS_ROOT / "quantile_stability_by_regime.csv")
quantile_stability

,group,full_period_spread,chron_block=0_spread,chron_block=1_spread,chron_block=2_spread,chron_block=3_spread,chron_block_sign_consistent,vol_regime=low_vol_spread,vol_regime=high_vol_spread,vol_regime_sign_consistent,trend_regime=bear_spread,trend_regime=bull_spread,trend_regime_sign_consistent
feature,,,,,,,,,,,,,
abs_return_1w,A,0.000575,0.001186,0.000091,0.000389,0.000636,True,0.000593,0.000556,True,0.001277,0.000316,True
abs_return_2w,A,0.000258,-0.000143,0.000761,0.000701,-0.000287,False,0.000541,-0.000041,False,0.000152,0.000304,True
abs_return_4w,A,0.000693,0.002071,-0.000171,0.000494,0.000403,False,0.000230,0.001164,True,0.001686,0.000316,True
abs_return_8w,A,0.000676,0.000922,0.000439,0.000655,0.000699,True,0.000359,0.001003,True,0.001216,0.000475,True
abs_return_12w,A,0.000175,-0.000887,0.000667,0.000396,0.000474,False,0.000749,-0.000394,False,0.000414,0.000085,True
abs_return_26w,A,0.000120,0.001472,-0.000045,0.000120,-0.000941,False,-0.000725,0.000959,False,0.001681,-0.000461,False
excess_return_vs_spy_1w,B,0.000575,0.001186,0.000091,0.000389,0.000636,True,0.000593,0.000556,True,0.001277,0.000316,True
excess_return_vs_spy_2w,B,0.000258,-0.000143,0.000761,0.000701,-0.000287,False,0.000541,-0.000041,False,0.000152,0.000304,True
excess_return_vs_spy_4w,B,0.000693,0.002071,-0.000171,0.000494,0.000403,False,0.000230,0.001164,True,0.001686,0.000316,True


## Stage 6: redundancy tests

**Different design than Stages 2-4 on purpose**: those stages deliberately isolated
cross-sectional signal from time-series drift (that's what "IC" means). Redundancy is
the opposite question — "if a model sees these two columns together, row by row, how
much do they overlap" — and a model sees the full panel exactly as stored, cross-section
and time-series variation both included. So this stage correlates on the **full,
unrestricted `research` panel** (all `weeks x symbols` rows, pairwise-complete), and
**all 36 features including groups H/I** are back in scope — H/I's zero cross-sectional
variance was disqualifying for a cross-sectional rank test, but it's irrelevant here;
their time-series variation is exactly what a redundancy check needs to see.

Two different correlation types, deliberately:
- **Spearman** correlation matrix — the "highest correlations with other features" per
  feature, and a flagged list of near-duplicate pairs.
- **VIF** (variance inflation factor) — computed from the **Pearson** correlation matrix
  of the complete-case rows, since VIF is defined from OLS R² (a linear/Pearson
  concept), not from rank correlation. `VIF_i = 1 / (1 - R^2_i)` regressing feature `i`
  on every other feature; computed here via the standard identity
  `VIF = diag(inverse(Pearson correlation matrix))` rather than running 36 separate
  regressions — same result, one matrix inversion. A near-singular correlation matrix
  (expected here, see below) is handled with a pseudo-inverse rather than crashing.

In [17]:
spearman_corr = research[FEATURE_COLUMNS].corr(method="spearman")
spearman_corr.to_csv(OUTPUTS_ROOT / "spearman_correlation_matrix.csv")

TOP_N_CORRELATIONS = 5
top_corr_rows = []
for col in FEATURE_COLUMNS:
    others = spearman_corr[col].drop(index=col).sort_values(key=abs, ascending=False)
    for rank, (other_feature, corr) in enumerate(others.head(TOP_N_CORRELATIONS).items(), start=1):
        top_corr_rows.append({"feature": col, "rank": rank, "other_feature": other_feature, "spearman_corr": corr})

top_correlations = pd.DataFrame(top_corr_rows)
top_correlations.to_csv(OUTPUTS_ROOT / "top_correlations_per_feature.csv", index=False)

print(f"{spearman_corr.shape[0]}x{spearman_corr.shape[0]} Spearman correlation matrix saved")
top_correlations[top_correlations["feature"] == "excess_return_vs_spy_4w"]

36x36 Spearman correlation matrix saved


,feature,rank,other_feature,spearman_corr
40,excess_return_vs_spy_4w,1,cross_sectional_rank_4w,-0.911282
41,excess_return_vs_spy_4w,2,excess_return_vs_spy_2w,0.666469
42,excess_return_vs_spy_4w,3,excess_return_vs_spy_8w,0.666015
43,excess_return_vs_spy_4w,4,cross_sectional_rank_8w,-0.607567
44,excess_return_vs_spy_4w,5,abs_return_4w,0.559863


### Near-duplicate pairs (|Spearman| > 0.9)

In [18]:
DUPLICATE_THRESHOLD = 0.9

pairs = []
for i, col_i in enumerate(FEATURE_COLUMNS):
    for col_j in FEATURE_COLUMNS[i + 1 :]:
        corr = spearman_corr.loc[col_i, col_j]
        if abs(corr) > DUPLICATE_THRESHOLD:
            pairs.append({"feature_1": col_i, "feature_2": col_j, "spearman_corr": corr})

near_duplicates = pd.DataFrame(pairs).sort_values("spearman_corr", key=abs, ascending=False).reset_index(drop=True)
near_duplicates.to_csv(OUTPUTS_ROOT / "near_duplicate_feature_pairs.csv", index=False)

print(f"{len(near_duplicates)} feature pairs with |Spearman correlation| > {DUPLICATE_THRESHOLD}")
near_duplicates

10 feature pairs with |Spearman correlation| > 0.9


,feature_1,feature_2,spearman_corr
0,spy_return_1w,average_sector_return_1w,0.963165
1,trend_dist_from_ma_8w,trend_dist_from_ma_12w,0.924758
2,abs_return_2w,trend_dist_from_ma_4w,0.919722
3,abs_return_8w,trend_dist_from_ma_12w,0.912645
4,excess_return_vs_spy_4w,cross_sectional_rank_4w,-0.911282
5,realized_vol_8w,realized_vol_12w,0.908537
6,excess_return_vs_spy_8w,cross_sectional_rank_8w,-0.908008
7,abs_return_4w,trend_dist_from_ma_8w,0.906974
8,excess_return_vs_spy_1w,cross_sectional_rank_1w,-0.906345
9,excess_return_vs_spy_12w,cross_sectional_rank_12w,-0.906178


### VIF (variance inflation factor)

In [19]:
complete_case = research[FEATURE_COLUMNS].dropna()
pearson_corr = complete_case.corr(method="pearson")

# VIF_i = diag(inverse(Pearson correlation matrix))_i -- equivalent to regressing each
# feature on all others and taking 1/(1-R^2), without running 36 separate OLS fits.
# pinv (not inv) because several pairs are expected to be near-collinear by construction
# (e.g. excess_return_vs_spy_Nw = abs_return_Nw - a common market term), which makes the
# matrix ill-conditioned or singular -- a hard inverse would raise, not just report a
# large number.
vif_values = np.diag(np.linalg.pinv(pearson_corr.values))
vif = pd.Series(vif_values, index=FEATURE_COLUMNS, name="vif").sort_values(ascending=False)
vif_table = vif.to_frame()
vif_table["group"] = [feature_to_group.get(f, "-") for f in vif_table.index]

vif_table.to_csv(OUTPUTS_ROOT / "vif.csv")
print(f"VIF computed on {len(complete_case):,} complete-case rows (of {len(research):,} total)")
vif_table.head(15)

VIF computed on 12,254 complete-case rows (of 12,540 total)


,vif,group
trend_dist_from_ma_12w,97.063225,E
trend_dist_from_ma_8w,49.870679,-
average_sector_return_1w,44.333197,I
trend_dist_from_ma_26w,30.251421,E
trend_dist_from_ma_4w,27.303102,E
spy_return_1w,18.818829,H
realized_vol_12w,18.766531,D
abs_return_8w,15.222723,A
drawdown_dist_from_high_13w,15.077853,F
drawdown_dist_from_high_26w,15.050878,F


### Findings

**Surprise, verified directly**: `abs_return_Nw` vs. `excess_return_vs_spy_Nw` at the
*same* N — the pair that looks algebraically closest to duplicated
(`excess = abs - spy_return`) — is only **~0.55** correlated, not ~1.0, at every
horizon. Not a bug: `spy_return_Nw` is constant *within* a week (doesn't touch the
cross-sectional ordering at all) but varies a lot *across* weeks, and that between-week
variance is a large share of `abs_return_Nw`'s total pooled variance. Subtracting it
removes that shared component, leaving a genuinely different pooled signal even though
the two features rank sectors identically within any single week. So despite looking
redundant by construction, A and B are not close to redundant in the way this panel
would actually be modeled (pooled, not per-week) — worth keeping both.

**What actually is near-duplicated** (|Spearman| > 0.9): `spy_return_1w` /
`average_sector_return_1w` (0.96 — expected, SPY is close to a cap-weighted average of
these same 11 sectors), adjacent-window trend/momentum pairs
(`trend_dist_from_ma_8w`/`12w`, `abs_return_2w`/`trend_dist_from_ma_4w`,
`abs_return_8w`/`trend_dist_from_ma_12w`, `realized_vol_8w`/`12w`), and — as expected
from Stage 2's sign-mirroring — each `excess_return_vs_spy_Nw` vs. its own
`cross_sectional_rank_Nw` (~-0.91). The adjacent-window pairs are the more actionable
finding: within the trend and volatility families, neighboring windows (8w/12w) carry
largely the same information, so a model likely doesn't need every window from every
family.

**VIF**: `trend_dist_from_ma_12w` (97), `trend_dist_from_ma_8w` (50), and
`average_sector_return_1w` (44) stand out as almost fully linearly predictable from the
rest of the feature set combined — consistent with the trend family's internal
redundancy above. Most group A/B/C momentum features sit in a much more moderate 3-19
range; `sector_return_dispersion_1w` and `sectors_outperforming_spy_1w` (group I) have
the lowest VIF (1.6-1.7), meaning they add close to independent information relative to
everything else, despite showing weak/undefined standalone predictive power in Stages
2-4. No feature is dropped here, per the standing instruction — this is the input Stage
7's incremental ablation needs to decide whether a highly-collinear feature earns its
place once its whole family is already in the model.

## Stage 7: incremental model tests (walk-forward by year)

**Model**: `sklearn.linear_model.LogisticRegression`, the linear baseline named back in
Stage 6's VIF discussion. **Label**: `next_week_excess_return > 0` (the classification
target named at the very start of this diagnostics phase — the continuous label drove
every stage through 6, this is its first use). **Split**: expanding-window walk-forward
by calendar year — train on every complete year strictly before the test year, test on
that year, roll forward one year at a time. `MIN_TRAIN_YEARS = 3` (first test year is
the 4th calendar year present in the data) — few enough training years would make an
early fold's coefficients meaningless noise; this is a judgment call, stated explicitly
rather than left implicit.

**Row set**: the same complete-case rows for every feature-set step (the Stage 6
`complete_case` filter — anything with a defined `26w` feature, which every step from
the group-B baseline onward already needs), so the ablation comparison is about feature
power only, never about a shifting sample.

**Feature scaling**: `StandardScaler` fit on each fold's *training* rows only, applied
to both train and test — fit-on-test would leak that year's own distribution back into
the "unseen" evaluation.

**Feature sets, in the specified order**: B (relative momentum alone) -> A+B -> A+B+C
-> ... -> A+B+C+D+E+F+G+H+I (everything). Group H/I are valid model inputs here even
though they were unusable for the cross-sectional tests in Stages 2-4 — a pooled
row-level classifier doesn't need cross-sectional variance the way a within-week rank
correlation does.

**Two evaluation angles per feature set**, pooling every fold's held-out predictions
before scoring (not averaging per-year metrics, which would be noisy at ~572 rows/year):
- Classification: accuracy and ROC-AUC on `predict_proba`.
- Portfolio-relevant: rank each held-out week's 11 sectors by predicted probability,
  form top-K/bottom-K spreads for K=2 and K=3 (same construction and same K values as
  Stage 4's rank-and-hold, but ranked by the *model's* score instead of one raw
  feature) — this is the number that actually answers "does adding this feature group
  improve the thing we'd trade," not just classification accuracy in the abstract.

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

model_data = research.dropna(subset=FEATURE_COLUMNS).copy()
model_data["label_binary"] = (model_data["next_week_excess_return"] > 0).astype(int)
model_data["year"] = model_data["date"].dt.year

MIN_TRAIN_YEARS = 3
years = sorted(model_data["year"].unique())
test_years = years[MIN_TRAIN_YEARS:]

FEATURE_SET_STEPS = [
    ("B", FEATURE_GROUPS["B"]),
    ("A+B", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"]),
    ("A+B+C", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"]),
    ("A+B+C+D", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"] + FEATURE_GROUPS["D"]),
    ("A+B+C+D+E", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"] + FEATURE_GROUPS["D"] + FEATURE_GROUPS["E"]),
    ("A+B+C+D+E+F", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"] + FEATURE_GROUPS["D"] + FEATURE_GROUPS["E"] + FEATURE_GROUPS["F"]),
    ("A+B+C+D+E+F+G", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"] + FEATURE_GROUPS["D"] + FEATURE_GROUPS["E"] + FEATURE_GROUPS["F"] + FEATURE_GROUPS["G"]),
    ("A+B+C+D+E+F+G+H", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"] + FEATURE_GROUPS["D"] + FEATURE_GROUPS["E"] + FEATURE_GROUPS["F"] + FEATURE_GROUPS["G"] + FEATURE_GROUPS["H"]),
    ("A+B+C+D+E+F+G+H+I", FEATURE_GROUPS["A"] + FEATURE_GROUPS["B"] + FEATURE_GROUPS["C"] + FEATURE_GROUPS["D"] + FEATURE_GROUPS["E"] + FEATURE_GROUPS["F"] + FEATURE_GROUPS["G"] + FEATURE_GROUPS["H"] + FEATURE_GROUPS["I"]),
]

print(f"{len(model_data):,} complete-case rows, years {years[0]}-{years[-1]}")
print(f"walk-forward test years ({len(test_years)} folds): {test_years[0]}-{test_years[-1]}")
print(f"{len(FEATURE_SET_STEPS)} incremental feature sets, from {len(FEATURE_SET_STEPS[0][1])} to {len(FEATURE_SET_STEPS[-1][1])} features")

12,254 complete-case rows, years 2005-2026
walk-forward test years (19 folds): 2008-2026
9 incremental feature sets, from 6 to 34 features


In [21]:
def top_bottom_spread_from_scores(frame: pd.DataFrame, score_col: str, k: int) -> pd.Series:
    """One week's top-K/bottom-K next-week excess return, ranked by score_col (e.g. predicted prob)."""
    ranked = frame[score_col].rank(ascending=False, method="first")  # 1 = highest score
    n = len(frame)
    top_return = frame.loc[ranked <= k, "next_week_excess_return"].mean()
    bottom_return = frame.loc[ranked > n - k, "next_week_excess_return"].mean()
    return pd.Series({"top_return": top_return, "bottom_return": bottom_return})


def walk_forward_evaluate(feature_cols: list[str]) -> dict:
    """Expanding-window walk-forward by year; returns pooled out-of-fold predictions + metrics."""
    oof_frames = []
    for test_year in test_years:
        train_mask = model_data["year"] < test_year
        test_mask = model_data["year"] == test_year

        scaler = StandardScaler().fit(model_data.loc[train_mask, feature_cols])
        X_train = scaler.transform(model_data.loc[train_mask, feature_cols])
        X_test = scaler.transform(model_data.loc[test_mask, feature_cols])
        y_train = model_data.loc[train_mask, "label_binary"]

        clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
        predicted_proba = clf.predict_proba(X_test)[:, 1]

        fold = model_data.loc[test_mask, ["date", "symbol", "next_week_excess_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

    oof = pd.concat(oof_frames, ignore_index=True)

    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])

    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()

    return {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), **spread_stats}


ablation_rows = []
for label, feature_cols in FEATURE_SET_STEPS:
    metrics = walk_forward_evaluate(feature_cols)
    ablation_rows.append({"feature_set": label, "n_features": len(feature_cols), **metrics})
    print(f"{label}: accuracy={metrics['accuracy']:.4f}, auc={metrics['auc']:.4f}, "
          f"k3_spread={metrics['mean_spread_k3']:.5f}")

ablation_results = pd.DataFrame(ablation_rows)
ablation_results.to_csv(OUTPUTS_ROOT / "ablation_walk_forward_results.csv", index=False)
ablation_results

B: accuracy=0.4975, auc=0.4994, k3_spread=-0.00002


A+B: accuracy=0.4943, auc=0.4989, k3_spread=0.00005


A+B+C: accuracy=0.5042, auc=0.5034, k3_spread=0.00004


A+B+C+D: accuracy=0.5007, auc=0.5012, k3_spread=-0.00004


A+B+C+D+E: accuracy=0.4995, auc=0.5010, k3_spread=0.00002


A+B+C+D+E+F: accuracy=0.5031, auc=0.5015, k3_spread=-0.00047


A+B+C+D+E+F+G: accuracy=0.5022, auc=0.5026, k3_spread=-0.00050


A+B+C+D+E+F+G+H: accuracy=0.5055, auc=0.5031, k3_spread=-0.00048


A+B+C+D+E+F+G+H+I: accuracy=0.5001, auc=0.5021, k3_spread=-0.00053


,feature_set,n_features,accuracy,auc,n_test_rows,mean_spread_k2,hit_rate_k2,mean_spread_k3,hit_rate_k3
0,B,6,0.497470,0.499356,10670,-0.000231,0.501031,-0.000018,0.502062
1,A+B,12,0.494283,0.498920,10670,-0.000372,0.498969,0.000046,0.498969
2,A+B+C,16,0.504217,0.503437,10670,0.000460,0.505155,0.000042,0.495876
3,A+B+C+D,19,0.500750,0.501177,10670,0.000784,0.494845,-0.000037,0.492784
4,A+B+C+D+E,22,0.499531,0.500958,10670,0.000518,0.483505,0.000024,0.488660
5,A+B+C+D+E+F,24,0.503093,0.501450,10670,-0.000718,0.465979,-0.000474,0.464948
6,A+B+C+D+E+F+G,27,0.502156,0.502632,10670,-0.000620,0.475258,-0.000502,0.482474
7,A+B+C+D+E+F+G+H,31,0.505530,0.503145,10670,-0.000583,0.475258,-0.000478,0.475258
8,A+B+C+D+E+F+G+H+I,34,0.500094,0.502149,10670,-0.000654,0.477320,-0.000531,0.477320


### Findings

**Honest result: the linear baseline is indistinguishable from chance, and stays that
way as feature groups are added.** Accuracy and AUC sit at ~0.494-0.506 across every
one of the 9 feature sets — never meaningfully above the 0.50 no-skill line, and not
monotonically improving as groups are added (`A+B` is briefly *worse* than `B` alone;
the best AUC, 0.5034, belongs to `A+B+C`, not the full 9-group set). The portfolio-style
spread metrics tell the same story from a different angle and add a genuinely bad sign:
`hit_rate_k2` *declines* as more groups are added — 50.1% at `B` alone down to 47.5-47.7%
once F/G/H/I join — meaning the added features are, if anything, making the logistic
regression's top-2 selection worse, not better.

This is consistent with everything found so far: Stage 2's ICs were all under 0.016,
Stage 3's spreads were on the order of 0.03-0.07% per week, and Stage 6 showed
substantial internal redundancy. A linear model built on a set of individually weak,
partially collinear features, walk-forward validated on ~572 rows/year, has very little
to work with — this result doesn't contradict the earlier stages, it's the direct
consequence of them.

**Caveat, stated rather than silently fixed**: `LogisticRegression`'s default L2
regularization (`C=1.0`) is unturned here — no hyperparameter search was run. A
different `C`, or per-fold cross-validated regularization, could shift these numbers
somewhat, though it's unlikely to overturn a result this close to 0.50 given how weak
the underlying univariate signal already was. Retuning `C` is a reasonable next step if
useful, but wasn't done without being asked, per how this diagnostics phase has been
run throughout.

**What this does and doesn't mean**: it doesn't mean the features are worthless — Stage
3's terciles and Stage 5's regime-conditional ICs both showed real, if weak, directional
signal that a *linear, unconditional* combination apparently can't extract cleanly
(possibly because the useful information is more in interactions — e.g. Stage 5's
finding that momentum features are more sign-stable in certain vol/trend regimes than
others — which a plain logistic regression cannot represent, but the eventual neural
network can). This result is the actual evidence for why this project isn't stopping at
a linear model.

## Stage 8: block-bootstrap confidence intervals

**Method**: a moving block bootstrap on each weekly spread series — resample
contiguous blocks of `BLOCK_SIZE` consecutive weeks (not individual weeks) with
replacement, concatenate to the series' original length, take the mean, repeat
`N_BOOTSTRAP` times, and report the empirical 2.5th/97.5th percentiles as the 95% CI.
Blocks (not iid weeks) are the point: weekly financial returns can carry short-range
autocorrelation, and resampling single weeks independently would understate the true
uncertainty by breaking that structure. `BLOCK_SIZE = 8` (~2 months) is a judgment call
in the absence of a formal autocorrelation-length estimate — checked below via the
series' own lag-1..4 autocorrelation before committing to it, rather than picked blind.

**What gets bootstrapped** (kept to the results that matter for a go/no-go decision,
not all ~60 series available across Stages 4 and 7):
- The **full model** (`A+B+C+D+E+F+G+H+I`) and the **baseline model** (`B` alone)
  walk-forward top-K/bottom-K spreads from Stage 7, K=2 and K=3 — these are the actual
  candidate live-portfolio constructions, so this is the number a go/no-go decision
  would actually rest on. Re-derived here (not refit) from the same walk-forward
  procedure, just returning the raw weekly spread series instead of only summary stats.
- The two single features whose Stage 4 rank-and-hold spread held up best at K=2/3 —
  `volume_change_1w` and `realized_vol_12w` — as a simple-feature comparison point,
  reusing `rank_hold_weekly_spread.csv` directly rather than recomputing.

**Reading the result**: a 95% CI that excludes 0 means that spread is statistically
distinguishable from noise at this sample size; a CI that straddles 0 means it isn't —
regardless of how the point estimate alone looks.

In [22]:
def walk_forward_oof_spreads(feature_cols: list[str]) -> dict[int, pd.Series]:
    """Same walk-forward procedure as Stage 7, returning the raw weekly top-K/bottom-K
    spread series (indexed by date) for each K in RANK_K_VALUES, instead of summary stats."""
    oof_frames = []
    for test_year in test_years:
        train_mask = model_data["year"] < test_year
        test_mask = model_data["year"] == test_year

        scaler = StandardScaler().fit(model_data.loc[train_mask, feature_cols])
        X_train = scaler.transform(model_data.loc[train_mask, feature_cols])
        X_test = scaler.transform(model_data.loc[test_mask, feature_cols])
        y_train = model_data.loc[train_mask, "label_binary"]

        clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
        fold = model_data.loc[test_mask, ["date", "symbol", "next_week_excess_return"]].copy()
        fold["predicted_proba"] = clf.predict_proba(X_test)[:, 1]
        oof_frames.append(fold)

    oof = pd.concat(oof_frames, ignore_index=True)
    spreads = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spreads[k] = (weekly["top_return"] - weekly["bottom_return"]).dropna()
    return spreads


full_model_spreads = walk_forward_oof_spreads(FEATURE_SET_STEPS[-1][1])
baseline_model_spreads = walk_forward_oof_spreads(FEATURE_SET_STEPS[0][1])

print(f"full model ({FEATURE_SET_STEPS[-1][0]}): {len(full_model_spreads[3])} weeks, "
      f"mean k3 spread = {full_model_spreads[3].mean():.5f}")
print(f"baseline model ({FEATURE_SET_STEPS[0][0]}): {len(baseline_model_spreads[3])} weeks, "
      f"mean k3 spread = {baseline_model_spreads[3].mean():.5f}")

full model (A+B+C+D+E+F+G+H+I): 970 weeks, mean k3 spread = -0.00053
baseline model (B): 970 weeks, mean k3 spread = -0.00002


### Block size check and the bootstrap function

In [23]:
for lag in range(1, 5):
    print(f"lag-{lag} autocorrelation, full-model k3 spread: {full_model_spreads[3].autocorr(lag=lag):.4f}")

BLOCK_SIZE = 8
N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260812


def block_bootstrap_ci(values: pd.Series, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                        ci: float = 0.95, seed: int = BOOTSTRAP_SEED) -> dict:
    """Moving block bootstrap on a 1-D series; returns the point estimate, CI bounds, and significance flag."""
    arr = values.to_numpy()
    n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(np.ceil(n / block_size))
    max_start = n - block_size

    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s : s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()

    lower, upper = np.percentile(boot_means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return {
        "point_estimate": arr.mean(),
        "ci_lower": lower,
        "ci_upper": upper,
        "n_weeks": n,
        "significant": bool(lower > 0 or upper < 0),
    }

lag-1 autocorrelation, full-model k3 spread: -0.0641
lag-2 autocorrelation, full-model k3 spread: 0.0000
lag-3 autocorrelation, full-model k3 spread: 0.0034
lag-4 autocorrelation, full-model k3 spread: -0.0938


### Confidence intervals

In [24]:
rank_hold_weekly = pd.read_csv(OUTPUTS_ROOT / "rank_hold_weekly_spread.csv", parse_dates=["date"])

targets: dict[str, pd.Series] = {}
for k in RANK_K_VALUES:
    targets[f"full_model_k{k}"] = full_model_spreads[k]
    targets[f"baseline_model_k{k}"] = baseline_model_spreads[k]
    for feature in ["volume_change_1w", "realized_vol_12w"]:
        series = rank_hold_weekly[(rank_hold_weekly["feature"] == feature) & (rank_hold_weekly["k"] == k)]
        targets[f"{feature}_k{k}"] = series.set_index("date")["spread"]

bootstrap_rows = []
for name, series in targets.items():
    result = block_bootstrap_ci(series)
    bootstrap_rows.append({"target": name, **result})

bootstrap_summary = pd.DataFrame(bootstrap_rows).sort_values("target")
bootstrap_summary.to_csv(OUTPUTS_ROOT / "block_bootstrap_ci.csv", index=False)
bootstrap_summary

,target,point_estimate,ci_lower,ci_upper,n_weeks,significant
1,baseline_model_k2,-0.000231,-0.001652,0.001072,970,False
5,baseline_model_k3,-0.000018,-0.001100,0.001101,970,False
0,full_model_k2,-0.000654,-0.001960,0.000765,970,False
4,full_model_k3,-0.000531,-0.001568,0.000667,970,False
3,realized_vol_12w_k2,0.000585,-0.000826,0.001938,1128,False
7,realized_vol_12w_k3,0.000564,-0.000548,0.001642,1128,False
2,volume_change_1w_k2,0.000665,-0.000341,0.001652,1139,False
6,volume_change_1w_k3,0.000578,-0.000318,0.001443,1139,False


### Findings

**None of the 8 spreads are statistically distinguishable from zero at 95% confidence
— every CI straddles 0.** This includes both the full walk-forward model and the
B-only baseline (K=2 and K=3 each), and the two individual features that looked most
promising in Stage 4 (`volume_change_1w`, `realized_vol_12w`, whose positive point
estimates were the strongest survivors of the Stage 4 reversal effect). Point estimates
range from -0.00065 to +0.00067 per week — a few basis points, dwarfed by each CI's own
width of roughly ±0.001 to ±0.002.

**Autocorrelation check, for the record**: lag-1 through lag-4 autocorrelation of the
full-model spread series was -0.06, 0.00, 0.00, -0.09 — close to white noise. The
`BLOCK_SIZE=8` choice turned out to be conservative rather than necessary; an iid
bootstrap would likely have given very similar intervals here. Worth knowing, not worth
redoing — a conservative choice that turns out unneeded doesn't invalidate the result.

**What this means for the plan overall**: this is the rigorous version of what Stage 7
already suggested informally — none of what's been tested through 8 stages (11 sector
ETFs, weekly rebalancing, this feature set, a linear model) has produced a top-K
portfolio effect that survives a proper uncertainty estimate. That's a legitimate,
important result on its own, not a reason to consider the diagnostics phase a failure:
it's exactly the evidence needed before deciding whether a neural network is likely to
find real nonlinear/interaction structure this linear analysis couldn't reach, or
whether this feature set and universe genuinely don't contain a tradeable weekly signal
at this concentration (top-2/3, 11 assets). Both are real possibilities this notebook
doesn't resolve on its own.